# 02a — DeepLabV3+ (ResNet-50)
**Paper:** Mohammad et al., iSpaRo 2024. DOI: 10.1109/iSpaRo60631.2024.10687827

## 0. Configuración e Imports

In [1]:
import sys, time
sys.path.append('.')
import torch, torch.nn as nn
import torchvision
import pandas as pd, numpy as np
from mars_utils_fast import (          # ← mars_utils_fast
    set_seed, get_or_create_split, build_dataloaders,
    run_multi_seed, append_benchmark_results,
    visualize_predictions, count_parameters,
    FocalDiceLoss, NUM_CLASSES, IGNORE_INDEX, SEED
)

set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
MODEL_NAME = "deeplabv3plus"


Device: cuda


## 1. Dataset y DataLoaders

In [2]:
df = pd.read_csv("processed/manifest_256.csv")  # ← manifest_256
df_train, df_val, df_test = get_or_create_split(df)
train_loader, val_loader, test_loader = build_dataloaders(
    df_train, df_val, df_test, batch_size=16, num_workers=4  # ← batch 16, workers 4
)
print(f"Batches — train: {len(train_loader)} val: {len(val_loader)} test: {len(test_loader)}")


✅ Split cargado desde processed\split_indices.pkl
Train: 16074 {'MSL': 15901, 'MER': 173}
Val  :  3443 {'MSL': 3443}
Test :  3448 {'MSL': 3448}
Batches — train: 1005 val: 216 test: 216


## 2. Definición del Modelo
Backbone ResNet-50 preentrenado en ImageNet. Clasificador adaptado a 4 clases.
Loss total = CE_main + 0.4 * CE_aux (salida auxiliar de ResNet).

In [3]:
def build_deeplabv3plus():
    model = torchvision.models.segmentation.deeplabv3_resnet50(
        weights=None,
        weights_backbone=torchvision.models.ResNet50_Weights.DEFAULT,
        num_classes=NUM_CLASSES,
        aux_loss=True,
    )
    # Reemplazar cabeza principal y auxiliar
    model.classifier[4]     = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
    return model

model = build_deeplabv3plus().to(device)
params_M = count_parameters(model)
print(f"Parámetros: {params_M:.2f}M")

# Verificar forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 256).to(device)
    out   = model(dummy)
    print(f"Output main: {out['out'].shape}  aux: {out['aux'].shape}")


Parámetros: 42.00M
Output main: torch.Size([2, 4, 256, 256])  aux: torch.Size([2, 4, 256, 256])


## 3. Entrenamiento (3 seeds)
SGD + PolynomialLR, loss CE con ignore_index=255, aux_weight=0.4

In [4]:
import torch.optim as optim

def criterion_fn():
    return nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

def optimizer_fn(params):
    return optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=1e-4)

def scheduler_fn(opt):
    return optim.lr_scheduler.PolynomialLR(opt, total_iters=80, power=0.9)

summary = run_multi_seed(
    model_fn      = build_deeplabv3plus,
    df_train      = df_train,
    df_val        = df_val,
    df_test       = df_test,
    criterion_fn  = criterion_fn,
    optimizer_fn  = optimizer_fn,
    scheduler_fn  = scheduler_fn,
    model_name    = MODEL_NAME,
    device        = device,
    seeds         = [42, 123, 7],
    num_epochs    = 80,
    patience      = 10,
    batch_size    = 16,          # ← subido a 16 (AMP lo permite)
    num_workers   = 4,
    aux_weight    = 0.4,
    use_amp       = True,        # ← nuevo parámetro
    n_per_mission = 700,         # ← subset estratificado ~2100 imágenes train
)


──────────────────────────────────────────────────
  Seed 42 | deeplabv3plus


c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)
c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)



  deeplabv3plus_seed42 | device=cuda | AMP=True
Ep   1/80 | loss=1.4041 mIoU=0.3016 | val=0.4034 | rock=0.0000
Ep   2/80 | loss=0.8083 mIoU=0.5146 | val=0.5056 | rock=0.0000
Ep   3/80 | loss=0.6387 mIoU=0.5704 | val=0.5063 | rock=0.0000
Ep   4/80 | loss=0.5937 mIoU=0.5742 | val=0.5648 | rock=0.0000
Ep   5/80 | loss=0.5437 mIoU=0.6035 | val=0.5587 | rock=0.0000
Ep   6/80 | loss=0.5062 mIoU=0.6125 | val=0.5933 | rock=0.0000
Ep   7/80 | loss=0.4761 mIoU=0.6231 | val=0.5789 | rock=0.0000
Ep   8/80 | loss=0.4343 mIoU=0.6409 | val=0.5898 | rock=0.0000
Ep   9/80 | loss=0.4424 mIoU=0.6298 | val=0.5934 | rock=0.0000
Ep  10/80 | loss=0.4084 mIoU=0.6486 | val=0.5787 | rock=0.0000
Ep  11/80 | loss=0.4343 mIoU=0.6351 | val=0.6009 | rock=0.0002
Ep  12/80 | loss=0.4160 mIoU=0.6523 | val=0.6028 | rock=0.0000
Ep  13/80 | loss=0.4175 mIoU=0.6544 | val=0.5959 | rock=0.0221
Ep  14/80 | loss=0.3438 mIoU=0.6839 | val=0.5908 | rock=0.0004
Ep  15/80 | loss=0.3728 mIoU=0.6957 | val=0.6043 | rock=0.0040
Ep  16

c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)
c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)



  deeplabv3plus_seed123 | device=cuda | AMP=True
Ep   1/80 | loss=1.4144 mIoU=0.2744 | val=0.3215 | rock=0.0000
Ep   2/80 | loss=0.8177 mIoU=0.5108 | val=0.4951 | rock=0.0000
Ep   3/80 | loss=0.6813 mIoU=0.5470 | val=0.5069 | rock=0.0000
Ep   4/80 | loss=0.5755 mIoU=0.5971 | val=0.4046 | rock=0.0000
Ep   5/80 | loss=0.5268 mIoU=0.6053 | val=0.5796 | rock=0.0000
Ep   6/80 | loss=0.5175 mIoU=0.6111 | val=0.5599 | rock=0.0000
Ep   7/80 | loss=0.4717 mIoU=0.6274 | val=0.5908 | rock=0.0000
Ep   8/80 | loss=0.4744 mIoU=0.6249 | val=0.5973 | rock=0.0000
Ep   9/80 | loss=0.4477 mIoU=0.6394 | val=0.5987 | rock=0.0000
Ep  10/80 | loss=0.4034 mIoU=0.6516 | val=0.5786 | rock=0.0025
Ep  11/80 | loss=0.3947 mIoU=0.6517 | val=0.6055 | rock=0.0018
Ep  12/80 | loss=0.3767 mIoU=0.6680 | val=0.5961 | rock=0.0000
Ep  13/80 | loss=0.3696 mIoU=0.6877 | val=0.6040 | rock=0.0012
Ep  14/80 | loss=0.3618 mIoU=0.6932 | val=0.6010 | rock=0.0002
Ep  15/80 | loss=0.3862 mIoU=0.7080 | val=0.6074 | rock=0.0044
Ep  1

c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)
c:\Users\potot\OneDrive\Documentos\DEEP\mars_utils_fast.py:139: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("mission", group_keys=False)



  deeplabv3plus_seed7 | device=cuda | AMP=True
Ep   1/80 | loss=1.3727 mIoU=0.3141 | val=0.3423 | rock=0.0000
Ep   2/80 | loss=0.7825 mIoU=0.5195 | val=0.4400 | rock=0.0000
Ep   3/80 | loss=0.6435 mIoU=0.5554 | val=0.5164 | rock=0.0000
Ep   4/80 | loss=0.5821 mIoU=0.5849 | val=0.5502 | rock=0.0000
Ep   5/80 | loss=0.4910 mIoU=0.6196 | val=0.5766 | rock=0.0000
Ep   6/80 | loss=0.4962 mIoU=0.6142 | val=0.5682 | rock=0.0000
Ep   7/80 | loss=0.4940 mIoU=0.6258 | val=0.5557 | rock=0.0000
Ep   8/80 | loss=0.4431 mIoU=0.6361 | val=0.5766 | rock=0.0000
Ep   9/80 | loss=0.4325 mIoU=0.6348 | val=0.6070 | rock=0.0000
Ep  10/80 | loss=0.4329 mIoU=0.6403 | val=0.5991 | rock=0.0000
Ep  11/80 | loss=0.4050 mIoU=0.6412 | val=0.6018 | rock=0.0000
Ep  12/80 | loss=0.3748 mIoU=0.6550 | val=0.6112 | rock=0.0000
Ep  13/80 | loss=0.3705 mIoU=0.6579 | val=0.6068 | rock=0.0000
Ep  14/80 | loss=0.3983 mIoU=0.6532 | val=0.6068 | rock=0.0091
Ep  15/80 | loss=0.3711 mIoU=0.6858 | val=0.6163 | rock=0.0012
Ep  16/

## 4. Guardado de resultados

In [5]:
append_benchmark_results(
    model_name    = MODEL_NAME,
    best_epoch    = summary["best_epoch_mean"],
    val_miou      = summary["mIoU_mean"],
    test_miou     = summary["mIoU_mean"],
    test_miou_std = summary["mIoU_std"],
    test_miou_ci95= summary["mIoU_ci95"],
    test_acc      = summary["pixel_acc_mean"],
    iou_soil      = summary["iou_soil_mean"],
    iou_bedrock   = summary["iou_bedrock_mean"],
    iou_sand      = summary["iou_sand_mean"],
    iou_bigrock   = summary["iou_big_rock_mean"],
    params_M      = params_M,
    train_time_s  = summary["train_time_mean"],
)


✅ Resultados en results\benchmark_results.csv


## 5. Evaluación y Visualización

In [ ]:
# Cargar mejor modelo (seed 42)
import torch
best_model = build_deeplabv3plus().to(device)
ckpt = torch.load(f"checkpoints/{MODEL_NAME}_seed42_best.pth", map_location=device)
best_model.load_state_dict(ckpt["model_state"])

visualize_predictions(best_model, df_test, device, n=5,
    save_path=f"results/{MODEL_NAME}_predictions.png")
print("Resumen final:")
print(f"  mIoU : {summary['mIoU_mean']:.4f} ± {summary['mIoU_std']:.4f}")
for c in ["soil","bedrock","sand","big_rock"]:
    print(f"  IoU({c}): {summary[f'iou_{c}_mean']:.4f} ± {summary[f'iou_{c}_std']:.4f}")
